# Board Meeting RAG — 02: Build / Sync Vector Index

Creates a Delta Sync Vector Search index over the chunks table. Embeddings are
computed server-side by `databricks-gte-large-en`, so we only index the `content`
column. Metadata columns (`entity`, `quarter`, `meeting_date`, `doc_type`,
`page_number`, `section`, `source_file`) are kept for post-retrieval filtering.

Re-running this notebook is safe: it creates the index if missing, or triggers
a sync if it already exists.

In [ ]:
%pip install --quiet databricks-vectorsearch
dbutils.library.restartPython()

In [ ]:
from databricks.vector_search.client import VectorSearchClient

ENDPOINT_NAME = "acme_rag_endpoint"  # Reuses if it exists, otherwise creates a STANDARD endpoint.
SOURCE_TABLE = "acme_holdings.documents.board_meetings_chunks"
INDEX_NAME = "acme_holdings.embeddings.board_meetings_index"
EMBEDDING_MODEL = "databricks-gte-large-en"
PRIMARY_KEY = "chunk_id"
EMBEDDING_SOURCE_COLUMN = "content"

vsc = VectorSearchClient(disable_notice=True)


## Ensure endpoint exists and is online

In [ ]:
import datetime

endpoints = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if ENDPOINT_NAME not in endpoints:
    print(f"Creating endpoint {ENDPOINT_NAME} (~10-15 min to provision)…")
    vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    # SDK quirk: wait_for_endpoint wants a timedelta, not an int.
    vsc.wait_for_endpoint(ENDPOINT_NAME, timeout=datetime.timedelta(seconds=900))
else:
    print(f"Endpoint {ENDPOINT_NAME} already exists (assumed ONLINE).")

## Create-or-sync the index

In [ ]:
existing = {i["name"] for i in vsc.list_indexes(ENDPOINT_NAME).get("vector_indexes", [])}

if INDEX_NAME not in existing:
    print(f"Creating index {INDEX_NAME}…")
    vsc.create_delta_sync_index(
        endpoint_name=ENDPOINT_NAME,
        index_name=INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",
        primary_key=PRIMARY_KEY,
        embedding_source_column=EMBEDDING_SOURCE_COLUMN,
        embedding_model_endpoint_name=EMBEDDING_MODEL,
    )
else:
    print(f"Index {INDEX_NAME} already exists. Triggering sync…")
    vsc.get_index(ENDPOINT_NAME, INDEX_NAME).sync()

## Wait for the index to be ready and smoke-test

In [ ]:
import time

idx = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)
for _ in range(60):
    desc = idx.describe()
    state = desc.get("status", {}).get("detailed_state", "")
    ready = desc.get("status", {}).get("ready", False)
    print(f"state={state} ready={ready}")
    if ready and "ONLINE" in state:
        break
    time.sleep(15)

In [ ]:
# Smoke test: retrieve top 3 chunks for a Q1 2026 query
results = idx.similarity_search(
    query_text="What were the key Q1 2026 performance and allocation decisions?",
    columns=["chunk_id", "entity", "quarter", "meeting_date", "source_file", "content"],
    filters={"quarter": "2026Q1"},
    num_results=3,
)
for row in results.get("result", {}).get("data_array", []):
    print(row[:5])
    print((row[5] or "")[:300])
    print("---")